# Week 84, Day 5: PyTorch: a first look

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w084d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Value:
    def __init__(self, data, parents=()):
        self.data, self.grad, self.parents = data, 0.0, parents

a = Value(3.0)
b = Value(4.0)
c = Value(a.data * b.data, parents=(a, b))
# dc/da is b, and dc/db is a: the chain rule starts here
a.grad, b.grad = b.data, a.data
print(c.data, a.grad, b.grad, [p.data for p in c.parents])

In [ ]:
class Value:
    def __init__(self, data, children=()):
        self.data, self.grad = float(data), 0.0
        self._prev = tuple(children)
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        order, seen = [], set()
        def visit(v):
            if id(v) not in seen:
                seen.add(id(v))
                for child in v._prev:
                    visit(child)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

x, w, b = Value(2.0), Value(-3.0), Value(1.0)
y = x * w + b          # -5.0
z = y * y              # 25.0, uses y twice
z.backward()
print(z.data, y.grad, w.grad, x.grad, b.grad)

In [ ]:
import math
import random

class V:
    def __init__(self, data):
        self.data, self.grad = data, 0.0

class Neuron:
    def __init__(self, n_in, rng):
        self.w = [V(rng.uniform(-1, 1)) for _ in range(n_in)]
        self.b = V(0.0)

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out, rng):
        self.neurons = [Neuron(n_in, rng) for _ in range(n_out)]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

rng = random.Random(0)
net = [Layer(3, 4, rng), Layer(4, 4, rng), Layer(4, 1, rng)]
print(sum(len(layer.parameters()) for layer in net), "parameters")

> **Key idea.** PyTorch provides tensors (NumPy-like, GPU-capable), autograd (`requires_grad=True`, then `loss.backward()` fills every `.grad`), `torch.nn` (layers, activations, losses, `nn.Module`) and `torch.optim` (SGD, Adam). Automatic differentiation records each operation's inputs and local derivative; `backward()` sets the output's gradient to 1 and calls each node's backward rule in reverse topological order, **adding** to gradients so values used twice get both contributions. Modules expose `parameters()`. Every training loop is: `zero_grad()` (gradients accumulate), forward, loss, `backward()`, `step()`. Predict under `torch.no_grad()` after `model.eval()`. This lesson's engine follows Andrej Karpathy's micrograd.

### Exercise: A value that remembers

Write a class `Value(data, children=())` with `data` (a float), `grad` (starting at `0.0`), `_prev` (the tuple of children) and `_backward` (a function doing nothing until an operation sets it). Support `+` and `*` between `Value`s and with plain numbers on **either** side (`__add__`, `__radd__`, `__mul__`, `__rmul__`; wrap numbers in `Value`), each setting the result's `_backward` so it **adds** the right amounts to its inputs' gradients. Then `backward()`: order the nodes so each comes after its children, set this node's `grad` to 1 and call every `_backward` in reverse order.

In [ ]:
class Value:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_add_mul():\n    \"local rules and the chain rule\"\n    x, w, b = Value(2.0), Value(-3.0), Value(1.0)\n    y = x * w + b\n    z = y * y\n    z.backward()\n    assert (z.data, y.grad, w.grad, x.grad, b.grad) == (25.0, -10.0, -20.0, 30.0, -10.0)\n\ndef test_numbers_and_reuse():\n    \"numbers on either side; reused values add up\"\n    a = Value(3.0)\n    out = 2 * a + a * a + 1\n    out.backward()\n    assert out.data == 16.0 and a.grad == 8.0\n    c = Value(5.0)\n    d = c + c + c\n    d.backward()\n    assert c.grad == 3.0 and isinstance(d._prev, tuple)")

### Exercise: More operations

Extend the given `Value` with `__neg__`, `__sub__` and `__rsub__`, `__pow__` (for a plain-number exponent: the derivative of `xⁿ` is `n·xⁿ⁻¹`), `__truediv__` and `__rtruediv__` (via `** -1`), and the methods `exp()`, `log()`, `tanh()` (derivative `1 − tanh²`) and `relu()` (derivative 1 where the result is positive, else 0). Build the new operations from the existing ones where you can.

In [ ]:
import math

class Value:
    def __init__(self, data, children=()):
        self.data, self.grad = float(data), 0.0
        self._prev, self._backward = tuple(children), lambda: None

    @staticmethod
    def _wrap(other):
        return other if isinstance(other, Value) else Value(other)

    def __add__(self, other):
        other = self._wrap(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = self._wrap(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def backward(self):
        order, seen = [], set()
        def visit(v):
            if id(v) not in seen:
                seen.add(id(v))
                for child in v._prev:
                    visit(child)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\n\ndef _check(f, x0):\n    v = Value(x0)\n    out = f(v)\n    out.backward()\n    h = 1e-6\n    numeric = (f(Value(x0 + h)).data - f(Value(x0 - h)).data) / (2 * h)\n    assert abs(v.grad - numeric) < 1e-5, (v.grad, numeric)\n    return out.data\n\ndef test_ops():\n    \"every derivative agrees with a numerical one\"\n    assert _check(lambda x: x - 3, 1.0) == -2.0\n    assert _check(lambda x: 10 - x * 2, 1.5) == 7.0\n    assert _check(lambda x: -x, 4.0) == -4.0\n    assert _check(lambda x: x ** 3, 2.0) == 8.0\n    assert _check(lambda x: x / 4, 2.0) == 0.5\n    assert _check(lambda x: 1 / x, 4.0) == 0.25\n    assert abs(_check(lambda x: x.exp(), 1.0) - math.e) < 1e-12\n    assert _check(lambda x: x.log(), 1.0) == 0.0\n    assert abs(_check(lambda x: x.tanh(), 0.5) - math.tanh(0.5)) < 1e-12\n    assert _check(lambda x: (x * 2).relu(), 1.5) == 3.0 and _check(lambda x: x.relu(), -2.0) == 0.0\n    assert abs(_check(lambda x: (x * x + 1).log() / x, 0.7) - math.log(1.49) / 0.7) < 1e-12")

### Exercise: Neurons, layers and a network

Using the given `Value`, write `Neuron(n_in, rng, activation="relu")` with weights `Value(rng.uniform(-1, 1))` (one per input, drawn in order) and a bias `Value(0.0)`; calling it on a list of inputs returns `Σ wᵢxᵢ + b` passed through `relu`, `tanh`, or nothing for `"linear"`. `Layer(n_in, n_out, rng, activation)` holds `n_out` neurons and returns a list of their outputs. `MLP(n_in, sizes, rng)` holds one layer per size, all `"relu"` except the last, which is `"linear"`; calling it returns the last layer's list, or the single `Value` if that layer has one neuron. All three have `parameters()`.

In [ ]:
import math
import random

class Value:
    def __init__(self, data, children=()):
        self.data, self.grad = float(data), 0.0
        self._prev, self._backward = tuple(children), lambda: None
    @staticmethod
    def _wrap(o):
        return o if isinstance(o, Value) else Value(o)
    def __add__(self, other):
        other = self._wrap(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    __radd__ = __add__
    def __mul__(self, other):
        other = self._wrap(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    __rmul__ = __mul__
    def _unary(self, value, slope):
        out = Value(value, (self,))
        def _backward():
            self.grad += slope * out.grad
        out._backward = _backward
        return out
    def tanh(self):
        t = math.tanh(self.data)
        return self._unary(t, 1 - t * t)
    def relu(self):
        r = max(0.0, self.data)
        return self._unary(r, 1.0 if r > 0 else 0.0)
    def backward(self):
        order, seen, stack = [], set(), [(self, False)]
        while stack:
            v, done = stack.pop()
            if done:
                order.append(v)
            elif id(v) not in seen:
                seen.add(id(v))
                stack.append((v, True))
                stack.extend((c, False) for c in v._prev)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

class Neuron:
    ___

class Layer:
    ___

class MLP:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\n\ndef test_modules():\n    \"parameters found by asking\"\n    rng = random.Random(0)\n    n = Neuron(3, rng, \"linear\")\n    rng2 = random.Random(0)\n    assert [w.data for w in n.w] == [rng2.uniform(-1, 1) for _ in range(3)] and n.b.data == 0.0\n    out = n([1.0, 2.0, 3.0])\n    assert abs(out.data - sum(w.data * x for w, x in zip(n.w, [1, 2, 3]))) < 1e-12\n    assert len(n.parameters()) == 4\n    net = MLP(3, [4, 4, 1], random.Random(1))\n    assert len(net.parameters()) == 41\n    y = net([0.5, -1.0, 2.0])\n    assert isinstance(y, Value)\n    y.backward()\n    assert any(p.grad != 0 for p in net.parameters())\n    two = MLP(2, [3, 2], random.Random(2))([1.0, 1.0])\n    assert isinstance(two, list) and len(two) == 2\n    r = Neuron(2, random.Random(3), \"relu\")([-5.0, -5.0])\n    assert r.data >= 0")

### Exercise: An optimiser

Write a class `SGD(params, lr)` with `zero_grad()` (set every parameter's `grad` to 0) and `step()` (move every parameter's `data` by `−lr × grad`). Give it the same shape as `torch.optim.SGD`: it keeps the list of parameters it was given.

In [ ]:
class P:
    def __init__(self, data, grad=0.0):
        self.data, self.grad = data, grad

class SGD:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sgd():\n    \"zero_grad and step\"\n    params = [P(1.0, 0.5), P(-2.0, -1.0)]\n    opt = SGD(params, lr=0.1)\n    opt.step()\n    assert [round(p.data, 6) for p in params] == [0.95, -1.9]\n    opt.zero_grad()\n    assert [p.grad for p in params] == [0.0, 0.0]\n    opt.step()\n    assert [round(p.data, 6) for p in params] == [0.95, -1.9]\n    assert opt.params is params or list(opt.params) == params")

### Exercise: The training loop

With the given `Value`, `MLP` and `SGD`, write `train(model, X, y, lr=0.05, epochs=100)`: a PyTorch-style loop that, each epoch, clears the gradients, computes the **mean squared error** of `model(x)` against each label (`Σ (pred − label)²` over the examples, divided by their number, as a `Value`), calls `backward()` on it and steps the optimiser. Return the list of the loss's `data` for each epoch.

In [ ]:
import math
import random

class Value:
    def __init__(self, data, children=()):
        self.data, self.grad = float(data), 0.0
        self._prev, self._backward = tuple(children), lambda: None
    @staticmethod
    def _wrap(o):
        return o if isinstance(o, Value) else Value(o)
    def __add__(self, other):
        other = self._wrap(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    __radd__ = __add__
    def __mul__(self, other):
        other = self._wrap(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    __rmul__ = __mul__
    def __neg__(self):
        return self * -1
    def __sub__(self, other):
        return self + (-self._wrap(other))
    def __pow__(self, n):
        out = Value(self.data ** n, (self,))
        def _backward():
            self.grad += n * self.data ** (n - 1) * out.grad
        out._backward = _backward
        return out
    def relu(self):
        out = Value(max(0.0, self.data), (self,))
        def _backward():
            self.grad += (1.0 if out.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out
    def backward(self):
        order, seen, stack = [], set(), [(self, False)]
        while stack:
            v, done = stack.pop()
            if done:
                order.append(v)
            elif id(v) not in seen:
                seen.add(id(v))
                stack.append((v, True))
                stack.extend((c, False) for c in v._prev)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

class Neuron:
    def __init__(self, n_in, rng, relu=True):
        self.w = [Value(rng.uniform(-1, 1)) for _ in range(n_in)]
        self.b, self.relu = Value(0.0), relu
    def __call__(self, x):
        total = sum((w * xi for w, xi in zip(self.w, x)), self.b)
        return total.relu() if self.relu else total
    def parameters(self):
        return self.w + [self.b]

class MLP:
    def __init__(self, n_in, sizes, rng):
        dims = [n_in] + list(sizes)
        self.layers = [[Neuron(dims[i], rng, i < len(sizes) - 1) for _ in range(dims[i + 1])] for i in range(len(sizes))]
    def __call__(self, x):
        for layer in self.layers:
            x = [n(x) for n in layer]
        return x[0] if len(x) == 1 else x
    def parameters(self):
        return [p for layer in self.layers for n in layer for p in n.parameters()]

class SGD:
    def __init__(self, params, lr):
        self.params, self.lr = params, lr
    def zero_grad(self):
        for p in self.params:
            p.grad = 0.0
    def step(self):
        for p in self.params:
            p.data -= self.lr * p.grad

def train(model, X, y, lr=0.05, epochs=100):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\n\ndef test_loop():\n    \"the loss falls\"\n    X = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]\n    y = [1.0, -1.0, -1.0, 1.0]\n    model = MLP(3, [4, 4, 1], random.Random(0))\n    losses = train(model, X, y, lr=0.05, epochs=100)\n    assert len(losses) == 100 and losses[-1] < losses[0] / 5 and losses[-1] < 0.1, (losses[0], losses[-1])\n    preds = [model(x).data for x in X]\n    assert all((p > 0) == (t > 0) for p, t in zip(preds, y))\n    again = train(MLP(3, [4, 4, 1], random.Random(0)), X, y, lr=0.05, epochs=100)\n    assert again == losses, \"the same seed gives the same training\"")

### Exercise: Why zero_grad?

Gradients **accumulate**. Using the given `Value`, write `grads_without_zeroing(x0, times)`: make `x = Value(x0)`, then `times` times build `y = x * x * 3` and call `y.backward()` **without** resetting `x.grad`; return `x.grad`. Then `grads_with_zeroing(x0, times)`: the same, but set `x.grad = 0.0` before each backward pass; return `x.grad`. (The true derivative of `3x²` is `6x`.)

In [ ]:
class Value:
    def __init__(self, data, children=()):
        self.data, self.grad = float(data), 0.0
        self._prev, self._backward = tuple(children), lambda: None
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def backward(self):
        order, seen = [], set()
        def visit(v):
            if id(v) not in seen:
                seen.add(id(v))
                for c in v._prev:
                    visit(c)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

def grads_without_zeroing(x0, times):
    ___

def grads_with_zeroing(x0, times):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_accumulate():\n    \"the sum of every pass, unless cleared\"\n    assert grads_with_zeroing(2.0, 1) == 12.0\n    assert grads_without_zeroing(2.0, 3) == 36.0\n    assert grads_with_zeroing(2.0, 3) == 12.0")

### Bug hunt: Half a gradient

The multiplication's backward rule **assigns** gradients instead of adding them, so for `a * a` the second path overwrites the first: it prints `3.0`, but `d(a²)/da` at 3 is `6.0`. Fix the rule so it prints `6.0`.

In [ ]:
class Value:
    def __init__(self, data, children=()):
        self.data, self.grad, self._prev, self._backward = data, 0.0, children, lambda: None
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad = other.data * out.grad
            other.grad = self.data * out.grad
        out._backward = _backward
        return out

a = Value(3.0)
out = a * a
out.grad = 1.0
out._backward()
print(a.grad)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"+=\"\n    expect_output(\"6.0\")")

### Bug hunt: Backwards in the wrong order

`backward` calls the nodes' rules in the order they were **visited** (children first), so each child runs before its parent has passed any gradient down: every gradient stays `0.0`. Run them in **reverse** so it prints `x.grad = -3.0`.

In [ ]:
class Value:
    def __init__(self, data, children=()):
        self.data, self.grad, self._prev, self._backward = data, 0.0, children, lambda: None
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def backward(self):
        order, seen = [], set()
        def visit(v):
            if id(v) not in seen:
                seen.add(id(v))
                for c in v._prev:
                    visit(c)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in order:
            v._backward()

x, w, k = Value(2.0), Value(-3.0), Value(1.0)
(x * w * k).backward()
print("x.grad =", x.grad)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"reverse topological order\"\n    expect_output(\"x.grad = -3.0\")")

### Bug hunt: Steps that keep growing

The loop never clears the gradient, so each step uses the sum of all the gradients so far and the parameter shoots past the answer: it prints a huge number. Clear the gradient at the start of each step so `w` settles at the minimum of `(w − 3)²`: `w = 3.0`.

In [ ]:
class P:
    def __init__(self, data):
        self.data, self.grad = data, 0.0

w = P(0.0)
for _ in range(100):
    w.grad += 2 * (w.data - 3)      # what loss.backward() would add
    w.data -= 0.1 * w.grad
print("w =", round(w.data, 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"zero_grad first\"\n    expect_output(\"w = 3.0\")")

### Puzzle: A PyTorch training loop

Arrange a real PyTorch training loop (to run on your own computer). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # for out = a * b with a = 3 and b = 5, the gradient a receives when out.grad is 1
q = ___   # for out = a + a, the gradient a receives when out.grad is 1
r = ___   # the PyTorch call that fills in every parameter's .grad
s = ___   # the optimiser call that must come first in each step, because gradients accumulate
t = ___   # how many parameters nn.Linear(2, 8) has

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 5 and q == 2, \"p and q\"\n    assert r in (\"loss.backward()\", \"backward()\", \"backward\") and s in (\"optimizer.zero_grad()\", \"zero_grad()\", \"zero_grad\") and t == 24, \"r, s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: Adam against SGD

Write `Adam(params, lr=0.05, betas=(0.9, 0.999), eps=1e-8)` with `zero_grad()` and `step()`. For each parameter it keeps `m` and `v` (starting at 0) and a step count `t` shared by all; each step does `t += 1`, then for every parameter `m = β₁m + (1 − β₁)g`, `v = β₂v + (1 − β₂)g²`, `m̂ = m / (1 − β₁ᵗ)`, `v̂ = v / (1 − β₂ᵗ)` and `data -= lr × m̂ / (√v̂ + eps)`. Then `fit(make_model, X, y, optimiser, epochs)`: build a model with `make_model()`, an optimiser with `optimiser(model.parameters())`, and train with the PyTorch loop on the **mean squared error** (as in the training-loop exercise); return `(model, losses)`. With the same model and epochs, Adam should end with a much lower loss than SGD at the same learning rate on the data in the checks.

In [ ]:
import math
import random

class Value:
    def __init__(self, data, children=()):
        self.data, self.grad = float(data), 0.0
        self._prev, self._backward = tuple(children), lambda: None
    @staticmethod
    def _wrap(o):
        return o if isinstance(o, Value) else Value(o)
    def __add__(self, other):
        other = self._wrap(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    __radd__ = __add__
    def __mul__(self, other):
        other = self._wrap(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    __rmul__ = __mul__
    def __sub__(self, other):
        return self + self._wrap(other) * -1
    def __pow__(self, n):
        out = Value(self.data ** n, (self,))
        def _backward():
            self.grad += n * self.data ** (n - 1) * out.grad
        out._backward = _backward
        return out
    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            self.grad += (1 - t * t) * out.grad
        out._backward = _backward
        return out
    def backward(self):
        order, seen, stack = [], set(), [(self, False)]
        while stack:
            v, done = stack.pop()
            if done:
                order.append(v)
            elif id(v) not in seen:
                seen.add(id(v))
                stack.append((v, True))
                stack.extend((c, False) for c in v._prev)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

class MLP:
    def __init__(self, n_in, n_hidden, rng):
        self.hidden = [([Value(rng.uniform(-1, 1)) for _ in range(n_in)], Value(0.0)) for _ in range(n_hidden)]
        self.out = ([Value(rng.uniform(-1, 1)) for _ in range(n_hidden)], Value(0.0))
    def __call__(self, x):
        h = [sum((w * xi for w, xi in zip(ws, x)), b).tanh() for ws, b in self.hidden]
        ws, b = self.out
        return sum((w * hi for w, hi in zip(ws, h)), b)
    def parameters(self):
        return [p for ws, b in self.hidden + [self.out] for p in ws + [b]]

class SGD:
    def __init__(self, params, lr=0.05):
        self.params, self.lr = params, lr
    def zero_grad(self):
        for p in self.params:
            p.grad = 0.0
    def step(self):
        for p in self.params:
            p.data -= self.lr * p.grad

class Adam:
    ___

def fit(make_model, X, y, optimiser, epochs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\nimport random\n\nX = [[x / 4] for x in range(-8, 9)]\nY = [math.sin(2 * x[0]) for x in X]\n\ndef test_adam_step():\n    \"the update rule\"\n    class P:\n        def __init__(self, data, grad):\n            self.data, self.grad = data, grad\n    p = P(1.0, 0.5)\n    opt = Adam([p], lr=0.1)\n    opt.step()\n    assert abs(p.data - 0.9) < 1e-6, \"the first Adam step moves by about lr\"\n    opt.zero_grad()\n    assert p.grad == 0.0\n    p.grad = 0.5\n    opt.step()\n    assert abs(p.data - 0.8) < 1e-6 and opt.t == 2\n\ndef test_compare():\n    \"Adam trains this faster\"\n    make = lambda: MLP(1, 8, random.Random(0))\n    sgd_model, sgd_losses = fit(make, X, Y, lambda ps: SGD(ps, lr=0.01), 150)\n    adam_model, adam_losses = fit(make, X, Y, lambda ps: Adam(ps, lr=0.01), 150)\n    assert len(sgd_losses) == len(adam_losses) == 150\n    assert adam_losses[-1] < adam_losses[0] / 5 and adam_losses[-1] < sgd_losses[-1] / 2, (sgd_losses[-1], adam_losses[-1])\n    assert abs(adam_model([0.5]).data - math.sin(1.0)) < 0.3")

## Warm-up

If every result remembers the values it was made from (its **parents**) and the local rule for its derivative, then gradients can be worked out automatically by walking that history backwards. That's all automatic differentiation is, and today you'll build it properly.

## What PyTorch adds

PyTorch's four main parts, each something you've already done by hand:

- **Tensors**: multi-dimensional arrays like NumPy's, with the same kind of operations and broadcasting, that can also live on a graphics card (GPU), where large matrix multiplications are many times faster.
- **Autograd**: a tensor created with `requires_grad=True` records every operation done with it. Calling `.backward()` on a result (the loss) fills in `.grad` for every such tensor, using the chain rule, exactly as yesterday's `backward` did, but for any program.
- **`torch.nn`**: layers (`nn.Linear`, `nn.Conv2d`, ...), activations (`nn.ReLU`), losses (`nn.BCEWithLogitsLoss`, `nn.MSELoss`) and `nn.Module`, the base class for models, which keeps track of their parameters.
- **`torch.optim`**: optimisers (`SGD`, `Adam`) that update every parameter from its gradient.

The training loop is the same four steps as yesterday. What changes is that you never write the backward pass.

## Automatic differentiation

Build it for single numbers. A `Value` holds `data`, a `grad` (starting at 0), the `Value`s it was computed from, and a small function `_backward` that knows the **local derivative** of the operation that made it. For `out = a + b`, both inputs get `out.grad` (the derivative of a sum with respect to each term is 1); for `out = a * b`, `a` gets `b.data × out.grad` and `b` gets `a.data × out.grad`. That multiplication by `out.grad` is the chain rule.

Two details matter:

- Gradients must be **added** (`+=`), not assigned: if a value is used twice (`a * a`, or one weight used for many examples), its gradient is the sum of both paths.
- `backward()` must call each node's `_backward` **after** every node that uses it has passed its gradient down: put the nodes in **topological order** (each after its parents), set the output's gradient to 1, and go through that order in reverse.

`dz/dy = 2y = −10`, so `dz/dw = −10 × x = −20`, `dz/dx = −10 × w = 30` and `dz/db = −10`. Nothing was written specially for this expression. This design follows **micrograd**, a tiny autograd engine by Andrej Karpathy (MIT licence), which today's exercises adapt.

## Modules and parameters

With `Value`, a neuron is a list of weight `Value`s and a bias `Value`; calling it on inputs builds the expression `activation(Σ wᵢxᵢ + b)`. A layer is a list of neurons; a network is a list of layers. Each provides **`parameters()`**, the list of every `Value` that training should change. PyTorch's `nn.Module` does the same job: any `nn.Linear` you assign as an attribute of your model is found by `model.parameters()`.

`(3 + 1) × 4 + (4 + 1) × 4 + (4 + 1) × 1 = 41`: every weight and bias, found by asking the modules.

## The training loop

In PyTorch every training loop has the same five lines inside it:

```python
for epoch in range(epochs):
    optimizer.zero_grad()            # 1. clear last step's gradients
    predictions = model(X)           # 2. forward pass (records the graph)
    loss = loss_fn(predictions, y)   # 3. one number to minimise
    loss.backward()                  # 4. fill in every parameter's .grad
    optimizer.step()                 # 5. update every parameter
```

Step 1 exists because gradients **accumulate** (`+=`, as in `Value`): without it, each step would use the sum of all previous gradients. An **optimiser** holds the list of parameters: `SGD` does `p.data -= lr * p.grad` for each; **`Adam`** keeps a running average of each parameter's gradient and of its square, and scales each step by them, which usually trains faster with less tuning of the learning rate.

After training, predict inside `with torch.no_grad():` (don't record a graph you won't use) and call `model.eval()` (some layers behave differently while training).

## On your own computer

Install PyTorch (`pip install torch`; the PyTorch website has a selector for your system, and the CPU version is fine for everything here), then run this, which is yesterday's XOR network with automatic gradients:

```python
import torch
from torch import nn

torch.manual_seed(0)
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))
loss_fn = nn.BCEWithLogitsLoss()          # sigmoid and cross-entropy in one stable step
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(500):
    optimizer.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    print(loss.item(), (torch.sigmoid(model(X)) > 0.5).int().ravel().tolist())

a = torch.tensor(2.0, requires_grad=True)
b = (a * a + 3 * a)
b.backward()
print(a.grad)                              # 2a + 3 = 7
```

For a model you write yourself, subclass `nn.Module`:

```python
class Net(nn.Module):
    def __init__(self, n_in, n_hidden):
        super().__init__()
        self.hidden = nn.Linear(n_in, n_hidden)
        self.out = nn.Linear(n_hidden, 1)

    def forward(self, x):
        return self.out(torch.relu(self.hidden(x)))
```

Bigger data goes through a `torch.utils.data.DataLoader`, which hands out shuffled mini-batches; `model.to("cuda")` and `X.to("cuda")` move work to a graphics card if you have a suitable one; `torch.save(model.state_dict(), "model.pt")` saves the learned weights.

## Practice

## Quiz

## Daily challenge

## Before you go

If you can, install PyTorch on your own computer and run the XOR program from "On your own computer". Then change it: remove `optimizer.zero_grad()` and watch what happens to the loss; swap `Adam` for `torch.optim.SGD` with the same learning rate; print `model[0].weight.grad` after a backward pass. Each experiment matches something you built today. Tomorrow is the Week 84 **Forge**: a complete machine-learning project, from raw text to an evaluated classifier.